# Simple Pair Backtest

A minimal notebook to backtest any two-ETF pair with periodic rebalancing.

## What this does
- Supports long/short direction per leg
- Rebalances both legs once per chosen period
- Sizes notionals to hit a target net beta exposure (default: beta-neutral)
- Tracks equity curve, drawdown, and simple performance stats

## Starter pair
- Leg A: `APLX` (short)
- Leg B: `APLZ` (short)
- Target: net beta exposure = 0


In [20]:
import io
import time
import ftplib
from pathlib import Path

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8")

In [ ]:
# --- Parameters ---
# Auto-select all bucket-3 inverse ETFs with negative beta from latest screener.
UNIVERSE_SOURCE_CSV = "../data/etf_screened_today.csv"

# Keep these for plotting cell compatibility; they will be set from first pair run.

START = "2024-01-01"
END = None

# Direction: +1 = long, -1 = short
# Requested setup: short inverse ETF and short underlying.
LEG_A_SIDE = -1
LEG_B_SIDE = -1

# Beta to underlying (or benchmark beta exposure) for each leg
# Used as manual fallback if auto lookup does not find a value.
# Inverse ETF leg defaults to -2x; underlying leg defaults to +1x.
LEG_A_BETA = -2.0
LEG_B_BETA = 1.0
USE_BETA_FROM_SCREENED = True
BETA_SOURCE_CSV = UNIVERSE_SOURCE_CSV

INITIAL_CAPITAL = 100_000
TARGET_GROSS_MULTIPLIER = 1.0   # gross notional as multiple of equity
TARGET_NET_BETA_NOTIONAL = 0.0  # 0 = beta-neutral

# Borrow settings (applies daily carry on short legs)
USE_BORROW_FROM_SCREENED = True
# ETF borrow: average fee-only borrow across all historical screener snapshots.
BORROW_SOURCE_GLOBS = [
    "../data/etf_screened_today.csv",
    "../data/runs/*/etf_screened_today.csv",
    "../../etf-dashboard/data/etf_screened_today.csv",
    "../../etf-dashboard/data/runs/*/etf_screened_today.csv",
]
# Underlying borrow: IBKR FTP snapshot (same method used in daily_screener.py).
USE_IBKR_FTP_FOR_UNDERLYINGS = True
FTP_HOST = "ftp2.interactivebrokers.com"
FTP_USER = "shortstock"
FTP_PASS = ""
FTP_FILE = "usa.txt"
FTP_MAX_RETRIES = 3
IBKR_BORROW_CACHE_PATH = Path("../data/borrow_cache.csv")

BORROW_FALLBACK_ANNUAL = 0.1
MANUAL_BORROW_OVERRIDE = {
    # "APLZ": 0.12,
    # "AMZN": 0.01,
}

# Short-sale proceeds credit (annualized): OBFR - 40 bps.
OBFR_ANNUAL = 0.041
SHORT_PROCEEDS_SPREAD_BPS = 40.0
SHORT_PROCEEDS_ANNUAL = max(0.0, OBFR_ANNUAL - SHORT_PROCEEDS_SPREAD_BPS / 10_000.0)

# Rebalance cadence comparison set.
REBALANCE_FREQ_MAP = {
    "bi_weekly": "2W-FRI",
}
# Default used by the existing plot cell after the comparison run.
REBALANCE_FREQ = "bi-weekly"

# Per-trade cost in bps applied to gross notional traded
FEE_BPS = 0.0

In [26]:
def _extract_close_series(raw: pd.DataFrame, ticker: str) -> pd.Series:
    """Return a 1D Close series across yfinance output shapes."""
    if raw is None or len(raw) == 0:
        return pd.Series(dtype=float, name=ticker)

    close = None
    if isinstance(raw.columns, pd.MultiIndex):
        # Common patterns:
        # - level0 has OHLC fields and level1 has ticker
        # - level0 has ticker and level1 has OHLC fields
        lvl0 = [str(x).lower() for x in raw.columns.get_level_values(0)]
        lvl1 = [str(x).lower() for x in raw.columns.get_level_values(1)]

        if "close" in lvl0:
            close = raw.xs("Close", axis=1, level=0)
        elif "close" in lvl1:
            close = raw.xs("Close", axis=1, level=1)
    else:
        if "Close" in raw.columns:
            close = raw["Close"]

    if close is None:
        raise RuntimeError(f"Missing Close column for {ticker}")

    # Ensure 1D output even if close came back as a 2D frame.
    if isinstance(close, pd.DataFrame):
        if close.shape[1] == 0:
            raise RuntimeError(f"Empty Close data for {ticker}")
        if ticker in close.columns:
            s = close[ticker]
        elif str(ticker).upper() in [str(c).upper() for c in close.columns]:
            pick = next(c for c in close.columns if str(c).upper() == str(ticker).upper())
            s = close[pick]
        else:
            s = close.iloc[:, 0]
    else:
        s = close

    return pd.to_numeric(s, errors="coerce").rename(ticker)


def load_prices(leg_a_ticker: str, leg_b_ticker: str, start: str, end: str | None = None) -> pd.DataFrame:
    # Pull each leg independently so we can explicitly align to the later
    # first-valid date (true common-live window for decay calculations).
    a_raw = yf.download(
        leg_a_ticker,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False,
    )
    b_raw = yf.download(
        leg_b_ticker,
        start=start,
        end=end,
        auto_adjust=True,
        progress=False,
    )

    a = _extract_close_series(a_raw, leg_a_ticker).rename("a_px")
    b = _extract_close_series(b_raw, leg_b_ticker).rename("b_px")

    first_a = a.dropna().index.min()
    first_b = b.dropna().index.min()
    if pd.isna(first_a) or pd.isna(first_b):
        raise RuntimeError(f"No valid price history for selected pair: {leg_a_ticker}/{leg_b_ticker}")

    aligned_start = max(first_a, first_b)
    if aligned_start > pd.Timestamp(start):
        print(
            f"[PAIR START] {leg_a_ticker}/{leg_b_ticker} aligned start moved "
            f"from {pd.Timestamp(start).date()} to {aligned_start.date()}"
        )

    px = pd.concat([a, b], axis=1)
    px = px.loc[px.index >= aligned_start].dropna()
    if px.empty:
        raise RuntimeError("No overlapping price data for selected pair after start alignment.")
    return px


def _norm_sym(x: str) -> str:
    return str(x).strip().upper()


def _pick_borrow_fee_only(row: pd.Series) -> float | None:
    # Match dashboard behavior: prefer fee-only borrow fields.
    for key in ("borrow_current", "borrow_fee_annual", "borrow_net_annual"):
        if key not in row.index:
            continue
        v = row.get(key)
        if pd.isna(v):
            continue
        try:
            return float(v)
        except Exception:
            continue
    return None


def _collect_screened_csv_paths(glob_patterns: list[str] | str) -> list[str]:
    patterns = [glob_patterns] if isinstance(glob_patterns, str) else list(glob_patterns)
    out: list[str] = []
    seen: set[str] = set()

    for pattern in patterns:
        if not pattern:
            continue
        for p in Path().glob(pattern):
            if not p.is_file():
                continue
            rp = str(p.resolve())
            if rp in seen:
                continue
            seen.add(rp)
            out.append(rp)

    out.sort()
    return out


def load_bucket3_pairs(screened_csv: str) -> list[tuple[str, str]]:
    df = pd.read_csv(screened_csv)
    cols = {c.lower(): c for c in df.columns}
    etf_col = cols.get("etf")
    und_col = cols.get("underlying")
    bucket_col = cols.get("bucket")
    beta_col = cols.get("beta")

    required = {"etf": etf_col, "underlying": und_col, "beta": beta_col}
    missing = [k for k, v in required.items() if v is None]
    if missing:
        raise ValueError(f"Missing required columns in {screened_csv}: {missing}")

    if bucket_col:
        tmp = df[[etf_col, und_col, bucket_col, beta_col]].copy()
    else:
        # Some snapshots only include screened rows with no explicit bucket column.
        tmp = df[[etf_col, und_col, beta_col]].copy()

    tmp[etf_col] = tmp[etf_col].astype(str).map(_norm_sym)
    tmp[und_col] = tmp[und_col].astype(str).map(_norm_sym)
    tmp[beta_col] = pd.to_numeric(tmp[beta_col], errors="coerce")

    m = tmp[beta_col].notna() & (tmp[beta_col] < 0) & tmp[etf_col].ne("") & tmp[und_col].ne("")
    if bucket_col:
        m = m & tmp[bucket_col].astype(str).str.lower().eq("bucket_3_inverse")

    sel = tmp.loc[m, [etf_col, und_col]].drop_duplicates()
    pairs = [(r[etf_col], r[und_col]) for _, r in sel.iterrows()]
    if not pairs:
        raise RuntimeError("No inverse negative-beta pairs found in screener.")
    return pairs


def _parse_ftp_text(text: str) -> pd.DataFrame:
    lines = [ln for ln in text.splitlines() if ln.strip()]
    header_idx = next((i for i, ln in enumerate(lines) if ln.startswith("#SYM|")), None)
    if header_idx is None:
        raise ValueError("Could not find header line '#SYM|'")
    header_cols = [c.strip().lstrip("#").lower() for c in lines[header_idx].split("|")]
    data_lines = lines[header_idx + 1 :]
    df = pd.read_csv(io.StringIO("\n".join(data_lines)), sep="|", header=None, engine="python")
    df = df.iloc[:, : min(len(header_cols), df.shape[1])]
    df.columns = header_cols[: df.shape[1]]
    return df.drop(columns=[c for c in df.columns if not c or str(c).startswith("unnamed")], errors="ignore")


def fetch_ibkr_shortstock_file(
    filename: str = FTP_FILE,
    max_retries: int = FTP_MAX_RETRIES,
    cache_path: Path = IBKR_BORROW_CACHE_PATH,
) -> pd.DataFrame:
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            print(f"[FTP] Connecting to {FTP_HOST} ... (attempt {attempt}/{max_retries})")
            ftp = ftplib.FTP(timeout=30)
            try:
                ftp.connect(FTP_HOST, 21)
                ftp.login(user=FTP_USER, passwd=FTP_PASS)
                ftp.set_pasv(True)
                buf = io.BytesIO()
                ftp.retrbinary(f"RETR {filename}", buf.write)
            finally:
                try:
                    ftp.quit()
                except Exception:
                    try:
                        ftp.close()
                    except Exception:
                        pass

            text = buf.getvalue().decode("utf-8", errors="ignore")
            df = _parse_ftp_text(text)
            cache_path.parent.mkdir(parents=True, exist_ok=True)
            df.to_csv(cache_path, index=False)
            print(f"[FTP] Parsed {len(df)} rows (cached to {cache_path})")
            return df
        except Exception as e:
            last_err = e
            if attempt < max_retries:
                time.sleep(2 ** attempt)

    if cache_path.exists():
        print(f"[FTP] All retries failed. Falling back to cache: {cache_path}")
        return pd.read_csv(cache_path)
    raise ConnectionError(f"IBKR FTP unavailable and no cache present. Last error: {last_err}")


def _parse_rate_to_decimal(x) -> float:
    if x is None:
        return np.nan
    s = str(x).strip()
    if s == "" or s.upper() in {"N/A", "NA", "NONE", "NULL"}:
        return np.nan
    s = s.replace("%", "").strip()
    try:
        return float(s) / 100.0
    except Exception:
        return np.nan


def get_ibkr_borrow_map(symbols: list[str]) -> dict[str, float]:
    symbols = list(dict.fromkeys([_norm_sym(s) for s in symbols if str(s).strip()]))
    if not symbols:
        return {}

    short_df = fetch_ibkr_shortstock_file(FTP_FILE)
    for req in ("sym", "feerate"):
        if req not in short_df.columns:
            raise ValueError(f"Expected '{req}' column; got: {list(short_df.columns)}")

    df = short_df.copy()
    df["sym"] = df["sym"].astype(str).str.upper().str.strip()
    df["borrow_fee_annual"] = df["feerate"].map(_parse_rate_to_decimal)

    agg = (
        df.groupby("sym", as_index=False)
        .agg(borrow_fee_annual=("borrow_fee_annual", "max"))
        .dropna(subset=["borrow_fee_annual"])
    )
    return {str(r["sym"]): float(r["borrow_fee_annual"]) for _, r in agg.iterrows() if str(r["sym"]) in symbols}


def load_pair_borrow_rates(
    ticker_a: str,
    ticker_b: str,
    screened_csv_globs: list[str] | str,
    use_screened: bool,
    fallback_annual: float,
    underlying_ibkr_map: dict[str, float] | None = None,
    manual_override: dict[str, float] | None = None,
) -> tuple[float, float]:
    ticker_a = _norm_sym(ticker_a)
    ticker_b = _norm_sym(ticker_b)
    rates = {ticker_a: float(fallback_annual), ticker_b: float(fallback_annual)}

    # ETF leg: use historical average from screener snapshots.
    if use_screened:
        vals_a: list[float] = []
        for path in _collect_screened_csv_paths(screened_csv_globs):
            try:
                df = pd.read_csv(path)
            except Exception:
                continue
            if "ETF" not in df.columns:
                continue
            df["ETF"] = df["ETF"].astype(str).str.upper().str.strip()
            for _, row in df.iterrows():
                if str(row.get("ETF", "")).upper().strip() != ticker_a:
                    continue
                borrow = _pick_borrow_fee_only(row)
                if borrow is not None and np.isfinite(borrow):
                    vals_a.append(float(borrow))
        if vals_a:
            rates[ticker_a] = float(np.mean(vals_a))

    # Underlying leg: use IBKR FTP map (daily_screener method).
    if underlying_ibkr_map is not None and ticker_b in underlying_ibkr_map and np.isfinite(underlying_ibkr_map[ticker_b]):
        rates[ticker_b] = float(underlying_ibkr_map[ticker_b])

    if manual_override:
        for k, v in manual_override.items():
            rates[_norm_sym(k)] = float(v)

    return rates[ticker_a], rates[ticker_b]


def load_beta_values(
    ticker_a: str,
    ticker_b: str,
    screened_csv: str,
    use_screened: bool,
    fallback_a: float,
    fallback_b: float,
) -> tuple[float, float]:
    beta_map = {_norm_sym(ticker_a): float(fallback_a), _norm_sym(ticker_b): float(fallback_b)}

    if use_screened:
        try:
            df = pd.read_csv(screened_csv)
            cols_lower = {c.lower(): c for c in df.columns}
            etf_col = cols_lower.get("etf")
            beta_col = cols_lower.get("beta")
            if etf_col and beta_col:
                df[etf_col] = df[etf_col].astype(str).str.upper().str.strip()
                tmp = df[[etf_col, beta_col]].dropna()
                for _, r in tmp.iterrows():
                    etf = str(r[etf_col])
                    if etf in beta_map:
                        beta_map[etf] = float(r[beta_col])
        except Exception as e:
            print(f"[WARN] Could not load betas from {screened_csv}: {e}")

    return beta_map[_norm_sym(ticker_a)], beta_map[_norm_sym(ticker_b)]


def solve_beta_hedged_notionals(
    gross_notional: float,
    side_a: float,
    side_b: float,
    beta_a: float,
    beta_b: float,
    target_net_beta_notional: float = 0.0,
) -> tuple[float, float]:
    a = float(side_a) * float(beta_a)
    b = float(side_b) * float(beta_b)
    g = float(gross_notional)
    t = float(target_net_beta_notional)

    denom = a - b
    if abs(denom) < 1e-12:
        n_a = 0.5 * g
        n_b = 0.5 * g
    else:
        n_a = (t - b * g) / denom
        n_b = g - n_a

    n_a = max(0.0, n_a)
    n_b = max(0.0, n_b)
    s = n_a + n_b
    if s <= 0:
        return 0.5 * g, 0.5 * g
    return g * n_a / s, g * n_b / s


def run_pair_backtest(
    prices: pd.DataFrame,
    rebal_freq: str = "W-FRI",
    initial_capital: float = 100_000,
    gross_multiplier: float = 1.0,
    side_a: float = -1.0,
    side_b: float = -1.0,
    beta_a: float = 2.0,
    beta_b: float = -2.0,
    target_net_beta_notional: float = 0.0,
    borrow_a_annual: float = 0.0,
    borrow_b_annual: float = 0.0,
    short_proceeds_annual: float = 0.0,
    fee_bps: float = 0.0,
) -> pd.DataFrame:
    bt = prices.copy()
    bt["rebalance"] = False

    rf = str(rebal_freq).upper().strip()
    if rf == "TWICE_WEEKLY":
        rebalance_dates = bt.resample("W-TUE").last().index.union(bt.resample("W-FRI").last().index)
    elif rf in REBALANCE_FREQ_MAP and REBALANCE_FREQ_MAP[rf].upper() != "TWICE_WEEKLY":
        rebalance_dates = bt.resample(REBALANCE_FREQ_MAP[rf]).last().index
    else:
        rebalance_dates = bt.resample(rebal_freq).last().index

    bt.loc[bt.index.isin(rebalance_dates), "rebalance"] = True
    bt.iloc[0, bt.columns.get_loc("rebalance")] = True

    a_sh = 0.0
    b_sh = 0.0
    cash = float(initial_capital)

    rows = []
    fee_rate = fee_bps / 10_000.0
    borrow_a_daily = float(borrow_a_annual) / 252.0
    borrow_b_daily = float(borrow_b_annual) / 252.0
    short_proceeds_daily = float(short_proceeds_annual) / 252.0

    for dt, row in bt.iterrows():
        ap = float(row["a_px"])
        bp = float(row["b_px"])

        a_pos_notional = a_sh * ap
        b_pos_notional = b_sh * bp

        borrow_cost = 0.0
        short_proceeds_credit = 0.0
        if a_pos_notional < 0:
            borrow_cost += abs(a_pos_notional) * borrow_a_daily
            short_proceeds_credit += abs(a_pos_notional) * short_proceeds_daily
        if b_pos_notional < 0:
            borrow_cost += abs(b_pos_notional) * borrow_b_daily
            short_proceeds_credit += abs(b_pos_notional) * short_proceeds_daily

        financing_pnl = short_proceeds_credit - borrow_cost
        cash += financing_pnl

        equity = cash + a_pos_notional + b_pos_notional

        if bool(row["rebalance"]):
            target_gross = max(0.0, float(gross_multiplier) * equity)
            n_a, n_b = solve_beta_hedged_notionals(
                gross_notional=target_gross,
                side_a=side_a,
                side_b=side_b,
                beta_a=beta_a,
                beta_b=beta_b,
                target_net_beta_notional=target_net_beta_notional,
            )

            target_a_pos = float(side_a) * n_a
            target_b_pos = float(side_b) * n_b

            delta_a = target_a_pos - a_pos_notional
            delta_b = target_b_pos - b_pos_notional

            traded_gross = abs(delta_a) + abs(delta_b)
            fee = traded_gross * fee_rate

            cash -= delta_a
            cash -= delta_b
            cash -= fee

            a_sh = target_a_pos / ap if ap > 0 else 0.0
            b_sh = target_b_pos / bp if bp > 0 else 0.0

            a_pos_notional = a_sh * ap
            b_pos_notional = b_sh * bp
            equity = cash + a_pos_notional + b_pos_notional

        beta_notional = side_a * beta_a * abs(a_pos_notional) + side_b * beta_b * abs(b_pos_notional)

        rows.append(
            {
                "date": dt,
                "a_px": ap,
                "b_px": bp,
                "cash": cash,
                "a_shares": a_sh,
                "b_shares": b_sh,
                "a_pos_notional": a_pos_notional,
                "b_pos_notional": b_pos_notional,
                "gross_notional": abs(a_pos_notional) + abs(b_pos_notional),
                "beta_notional": beta_notional,
                "borrow_cost": borrow_cost,
                "short_proceeds_credit": short_proceeds_credit,
                "financing_pnl": financing_pnl,
                "equity": equity,
                "rebalance": bool(row["rebalance"]),
            }
        )

    out = pd.DataFrame(rows).set_index("date")
    out["ret"] = out["equity"].pct_change().fillna(0.0)
    out["cum_return"] = out["equity"] / out["equity"].iloc[0] - 1.0
    out["drawdown"] = out["equity"].div(out["equity"].cummax()).sub(1.0)
    out["cum_borrow_cost"] = out["borrow_cost"].cumsum()
    out["cum_short_proceeds_credit"] = out["short_proceeds_credit"].cumsum()
    out["cum_financing_pnl"] = out["financing_pnl"].cumsum()
    return out


def perf_stats(bt: pd.DataFrame) -> pd.Series:
    n = len(bt)
    if n < 2:
        return pd.Series(dtype=float)

    total_return = bt["equity"].iloc[-1] / bt["equity"].iloc[0] - 1.0
    ann_factor = 252 / max(1, n - 1)
    cagr = (1 + total_return) ** ann_factor - 1 if total_return > -1 else np.nan

    vol = bt["ret"].std() * np.sqrt(252)
    sharpe = (bt["ret"].mean() * 252 / vol) if vol > 0 else np.nan
    max_dd = bt["drawdown"].min()

    return pd.Series(
        {
            "Total Return": total_return,
            "CAGR": cagr,
            "Annual Vol": vol,
            "Sharpe": sharpe,
            "Max Drawdown": max_dd,
            "Avg |Beta Notional|": bt["beta_notional"].abs().mean(),
            "Total Borrow Cost": bt["borrow_cost"].sum(),
            "Total Short Proceeds Credit": bt.get("short_proceeds_credit", pd.Series(dtype=float)).sum(),
            "Net Financing PnL": bt.get("financing_pnl", pd.Series(dtype=float)).sum(),
            "Rebalance Count": int(bt["rebalance"].sum()),
        }
    )

In [29]:
BUCKET3_PAIRS = load_bucket3_pairs(UNIVERSE_SOURCE_CSV)
print(f"Loaded {len(BUCKET3_PAIRS)} inverse negative-beta pairs from {UNIVERSE_SOURCE_CSV}")

# Pull all underlying borrow rates in one FTP snapshot (daily_screener method).
underlying_symbols = sorted({und for _, und in BUCKET3_PAIRS})
underlying_ibkr_map: dict[str, float] = {}
if USE_IBKR_FTP_FOR_UNDERLYINGS:
    try:
        underlying_ibkr_map = get_ibkr_borrow_map(underlying_symbols)
        print(f"IBKR borrow loaded for {len(underlying_ibkr_map)}/{len(underlying_symbols)} underlyings")
    except Exception as e:
        print(f"[WARN] Could not load IBKR underlying borrow map: {e}")

pair_results: dict[tuple[str, str, str], dict] = {}
summary_rows: list[dict] = []
side_map = {1: "LONG", -1: "SHORT"}

for freq_label, freq_value in REBALANCE_FREQ_MAP.items():
    print(f"\n=== Rebalance mode: {freq_label} ({freq_value}) ===")
    for etf_sym, und_sym in BUCKET3_PAIRS:
        prices_i = load_prices(etf_sym, und_sym, START, END)

        beta_a_i, beta_b_i = load_beta_values(
            etf_sym,
            und_sym,
            BETA_SOURCE_CSV,
            USE_BETA_FROM_SCREENED,
            LEG_A_BETA,
            LEG_B_BETA,
        )

        borrow_a_i, borrow_b_i = load_pair_borrow_rates(
            etf_sym,
            und_sym,
            BORROW_SOURCE_GLOBS,
            USE_BORROW_FROM_SCREENED,
            BORROW_FALLBACK_ANNUAL,
            underlying_ibkr_map=underlying_ibkr_map,
            manual_override=MANUAL_BORROW_OVERRIDE,
        )

        bt_i = run_pair_backtest(
            prices_i,
            rebal_freq=freq_value,
            initial_capital=INITIAL_CAPITAL,
            gross_multiplier=TARGET_GROSS_MULTIPLIER,
            side_a=LEG_A_SIDE,
            side_b=LEG_B_SIDE,
            beta_a=beta_a_i,
            beta_b=beta_b_i,
            target_net_beta_notional=TARGET_NET_BETA_NOTIONAL,
            borrow_a_annual=borrow_a_i,
            borrow_b_annual=borrow_b_i,
            short_proceeds_annual=SHORT_PROCEEDS_ANNUAL,
            fee_bps=FEE_BPS,
        )

        stats_i = perf_stats(bt_i)
        pair_results[(freq_label, etf_sym, und_sym)] = {
            "prices": prices_i,
            "bt": bt_i,
            "stats": stats_i,
            "beta_a": beta_a_i,
            "beta_b": beta_b_i,
            "borrow_a": borrow_a_i,
            "borrow_b": borrow_b_i,
        }

        summary_rows.append(
            {
                "rebalance": freq_label,
                "pair": f"{etf_sym}/{und_sym}",
                "total_return": stats_i.get("Total Return", np.nan),
                "cagr": stats_i.get("CAGR", np.nan),
                "max_drawdown": stats_i.get("Max Drawdown", np.nan),
                "sharpe": stats_i.get("Sharpe", np.nan),
                "avg_abs_beta_notional": stats_i.get("Avg |Beta Notional|", np.nan),
                "total_borrow_cost": stats_i.get("Total Borrow Cost", np.nan),
                "total_short_proceeds_credit": stats_i.get("Total Short Proceeds Credit", np.nan),
                "net_financing_pnl": stats_i.get("Net Financing PnL", np.nan),
                "borrow_etf_hist_avg": borrow_a_i,
                "borrow_underlying_ibkr": borrow_b_i,
                "short_proceeds_rate": SHORT_PROCEEDS_ANNUAL,
            }
        )

summary_df = pd.DataFrame(summary_rows)
summary_df = summary_df.sort_values(["rebalance", "total_return"], ascending=[True, False]).reset_index(drop=True)
display(summary_df)

# Aggregate comparison by rebalance cadence.
freq_compare = (
    summary_df.groupby("rebalance", as_index=False)
    .agg(
        pairs=("pair", "count"),
        avg_total_return=("total_return", "mean"),
        median_total_return=("total_return", "median"),
        avg_cagr=("cagr", "mean"),
        avg_sharpe=("sharpe", "mean"),
        avg_max_drawdown=("max_drawdown", "mean"),
        avg_net_financing_pnl=("net_financing_pnl", "mean"),
    )
    .sort_values("avg_total_return", ascending=False)
    .reset_index(drop=True)
)
display(freq_compare)

Loaded 44 inverse negative-beta pairs from ../data/etf_screened_today.csv
[FTP] Connecting to ftp2.interactivebrokers.com ... (attempt 1/3)
[FTP] Parsed 18941 rows (cached to ..\data\borrow_cache.csv)
IBKR borrow loaded for 29/29 underlyings

=== Rebalance mode: bi_weekly (2W-FRI) ===
[PAIR START] SDS/SPY aligned start moved from 2024-01-01 to 2024-01-02
[PAIR START] QID/QQQ aligned start moved from 2024-01-01 to 2024-01-02
[PAIR START] DXD/DIA aligned start moved from 2024-01-01 to 2024-01-02
[PAIR START] TWM/IWM aligned start moved from 2024-01-01 to 2024-01-02
[PAIR START] SCO/USO aligned start moved from 2024-01-01 to 2024-01-02
[PAIR START] MSTZ/MSTR aligned start moved from 2024-01-01 to 2024-09-18
[PAIR START] NVDQ/NVDA aligned start moved from 2024-01-01 to 2024-01-02
[PAIR START] BTCZ/IBIT aligned start moved from 2024-01-01 to 2024-07-10
[PAIR START] ETHD/ETHA aligned start moved from 2024-01-01 to 2024-07-23
[PAIR START] CRCD/CRCL aligned start moved from 2024-01-01 to 2025-

,rebalance,pair,total_return,cagr,max_drawdown,sharpe,avg_abs_beta_notional,total_borrow_cost,total_short_proceeds_credit,net_financing_pnl,borrow_etf_hist_avg,borrow_underlying_ibkr,short_proceeds_rate
0,bi_weekly,APLZ/APLD,0.260458,1.945318,-0.024839,3.568363,12949.891529,2661.828685,872.278086,-1789.550598,0.339556,0.004234,0.037
1,bi_weekly,MSTZ/MSTR,0.211921,0.132237,-0.283709,0.536372,15082.167162,31731.310831,6077.732286,-25653.578546,0.581919,0.002963,0.037
2,bi_weekly,TSLQ/TSLA,0.188594,0.079521,-0.190306,0.636766,9143.086572,4527.752339,9023.452021,4495.699682,0.049032,0.002500,0.037
3,bi_weekly,IREZ/IREN,0.124773,0.731020,-0.048533,2.888743,17416.469773,5779.220863,833.674439,-4945.546424,0.774380,0.003791,0.037
4,bi_weekly,NVDQ/NVDA,0.109121,0.046937,-0.047987,0.689569,8681.666335,16775.400392,8785.589540,-7989.810851,0.210582,0.003648,0.037
5,bi_weekly,CLSZ/CLSK,0.085461,0.804772,-0.005180,7.038487,11591.322623,437.113255,523.514092,86.400837,0.092025,0.004134,0.037
6,bi_weekly,SOXS/SOXX,0.083310,0.036075,-0.051974,0.429299,9584.412302,11149.973950,8794.605312,-2355.368638,0.150294,0.013843,0.037
7,bi_weekly,FNGD/XLK,0.072927,0.031666,-0.151689,0.393984,7402.954311,14222.462438,9371.193060,-4851.269378,0.208267,0.004149,0.037
8,bi_weekly,NBIZ/NBIS,0.063765,0.334381,-0.100321,0.926815,21221.465233,2447.701390,843.863074,-1603.838316,0.340392,0.006509,0.037
9,bi_weekly,BEZ/BE,0.048990,0.351635,-0.047772,1.795824,13414.458027,3510.644229,598.088414,-2912.555815,0.703089,0.002500,0.037


,rebalance,pairs,avg_total_return,median_total_return,avg_cagr,avg_sharpe,avg_max_drawdown,avg_net_financing_pnl
0,bi_weekly,44,-0.056876,-0.060211,0.036964,-1.177959,-0.128424,-5778.160524


In [ ]:
# Bar chart diagnostics by inverse ETF for the selected rebalance cadence.
if "summary_df" not in globals() or summary_df.empty:
    raise RuntimeError("Run the backtest cell first to generate summary_df.")

rebalance_alias = {
    "bi-weekly": "bi_weekly",
    "bi_weekly": "bi_weekly",
    "weekly": "weekly",
    "twice-weekly": "twice_weekly",
    "twice_weekly": "twice_weekly",
    "monthly": "monthly",
}
selected_rebalance = rebalance_alias.get(str(REBALANCE_FREQ).lower(), str(REBALANCE_FREQ))
if selected_rebalance not in set(summary_df["rebalance"].astype(str)):
    selected_rebalance = str(summary_df["rebalance"].iloc[0])
    print(f"[INFO] REBALANCE_FREQ not found in summary_df; using '{selected_rebalance}'")

screened = pd.read_csv(UNIVERSE_SOURCE_CSV)
cols = {c.lower(): c for c in screened.columns}

etf_col = cols.get("etf")
und_col = cols.get("underlying")
und_vol_col = cols.get("vol_underlying_annual")
gross_decay_col = cols.get("gross_decay_annual")
if not etf_col or not und_col:
    raise RuntimeError("UNIVERSE_SOURCE_CSV must include ETF and Underlying columns")
if not und_vol_col or not gross_decay_col:
    raise RuntimeError("UNIVERSE_SOURCE_CSV must include vol_underlying_annual and gross_decay_annual")

plot_df = summary_df.copy()
plot_df = plot_df[plot_df["rebalance"].astype(str) == selected_rebalance].copy()
if plot_df.empty:
    raise RuntimeError(f"No rows for rebalance cadence '{selected_rebalance}'")

plot_df[["etf", "underlying"]] = plot_df["pair"].str.split("/", n=1, expand=True)
plot_df["etf"] = plot_df["etf"].astype(str).str.upper().str.strip()
plot_df["underlying"] = plot_df["underlying"].astype(str).str.upper().str.strip()

meta = screened[[etf_col, und_col, und_vol_col, gross_decay_col]].copy()
meta.columns = ["etf", "underlying", "underlying_vol", "gross_decay"]
meta["etf"] = meta["etf"].astype(str).str.upper().str.strip()
meta["underlying"] = meta["underlying"].astype(str).str.upper().str.strip()

plot_df = plot_df.merge(meta, on=["etf", "underlying"], how="left")
plot_df["net_borrow_cost"] = pd.to_numeric(plot_df.get("total_borrow_cost"), errors="coerce") - pd.to_numeric(plot_df.get("total_short_proceeds_credit"), errors="coerce")
plot_df["net_borrow_cost_pct"] = plot_df["net_borrow_cost"] / float(INITIAL_CAPITAL)
plot_df["short_proceeds_pct"] = pd.to_numeric(plot_df.get("total_short_proceeds_credit"), errors="coerce") / float(INITIAL_CAPITAL)
plot_df["cagr"] = pd.to_numeric(plot_df.get("cagr"), errors="coerce")
plot_df["underlying_vol"] = pd.to_numeric(plot_df.get("underlying_vol"), errors="coerce")
plot_df["gross_decay"] = pd.to_numeric(plot_df.get("gross_decay"), errors="coerce")

plot_df = plot_df.sort_values("cagr", ascending=False).reset_index(drop=True)

x = np.arange(len(plot_df))
width = 0.16
fig, ax = plt.subplots(figsize=(max(12, len(plot_df) * 0.45), 6.5))

ax.bar(x - 2 * width, plot_df["cagr"], width, label="CAGR")
ax.bar(x - width, plot_df["net_borrow_cost_pct"], width, label="Net Borrow Cost / Initial")
ax.bar(x, plot_df["short_proceeds_pct"], width, label="Short Proceeds / Initial")
ax.bar(x + width, plot_df["underlying_vol"], width, label="Underlying Vol (annual)")
ax.bar(x + 2 * width, plot_df["gross_decay"], width, label="Gross Decay Score")

ax.set_title(f"Pair Diagnostics by Inverse ETF ({selected_rebalance})")
ax.set_ylabel("Decimal (annualized or ratio)")
ax.set_xticks(x)
ax.set_xticklabels(plot_df["etf"], rotation=75, ha="right")
ax.axhline(0.0, color="black", linewidth=1)
ax.grid(True, axis="y", alpha=0.25)
ax.legend(ncol=2)
plt.tight_layout()
plt.show()

display(
    plot_df[
        [
            "rebalance",
            "pair",
            "cagr",
            "net_borrow_cost",
            "total_short_proceeds_credit",
            "underlying_vol",
            "gross_decay",
        ]
    ]
)

In [30]:

# Keep variables for downstream plotting cell compatibility.
plot_freq = REBALANCE_FREQ if REBALANCE_FREQ in REBALANCE_FREQ_MAP else "weekly"
LEG_A_TICKER, LEG_B_TICKER = BUCKET3_PAIRS[5]
prices = pair_results[(plot_freq, LEG_A_TICKER, LEG_B_TICKER)]["prices"]
bt = pair_results[(plot_freq, LEG_A_TICKER, LEG_B_TICKER)]["bt"]
stats = pair_results[(plot_freq, LEG_A_TICKER, LEG_B_TICKER)]["stats"]
beta_a = pair_results[(plot_freq, LEG_A_TICKER, LEG_B_TICKER)]["beta_a"]
beta_b = pair_results[(plot_freq, LEG_A_TICKER, LEG_B_TICKER)]["beta_b"]
borrow_a = pair_results[(plot_freq, LEG_A_TICKER, LEG_B_TICKER)]["borrow_a"]
borrow_b = pair_results[(plot_freq, LEG_A_TICKER, LEG_B_TICKER)]["borrow_b"]
print(
    f"Plot pair: {side_map.get(LEG_A_SIDE, LEG_A_SIDE)} {LEG_A_TICKER} / "
    f"{side_map.get(LEG_B_SIDE, LEG_B_SIDE)} {LEG_B_TICKER} | rebalance={plot_freq}"
)
print(f"Short proceeds credit rate used: {SHORT_PROCEEDS_ANNUAL:.2%} (OBFR - 40bps)")

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)

# Equity + both ETF prices (secondary axis)
ax_eq = axes[0]
ax_px = ax_eq.twinx()

line_eq = ax_eq.plot(bt.index, bt["equity"], label="Equity", color="tab:blue", linewidth=2)
line_a = ax_px.plot(prices.index, prices["a_px"], label=f"{LEG_A_TICKER} Price", color="tab:orange", alpha=0.8)
line_b = ax_px.plot(prices.index, prices["b_px"], label=f"{LEG_B_TICKER} Price", color="tab:green", alpha=0.8)

ax_eq.set_title(f"Equity + Prices: {LEG_A_TICKER} vs {LEG_B_TICKER}")
ax_eq.set_ylabel("Equity ($)")
ax_px.set_ylabel("ETF Price ($)")
ax_eq.grid(True, alpha=0.3)

lines = line_eq + line_a + line_b
labels = [l.get_label() for l in lines]
ax_eq.legend(lines, labels, loc="upper left")

axes[1].plot(bt.index, bt["drawdown"] * 100)
axes[1].set_title("Drawdown")
axes[1].set_ylabel("Drawdown (%)")
axes[1].grid(True, alpha=0.3)

axes[2].plot(bt.index, bt["beta_notional"], label="Net Beta Notional")
axes[2].axhline(0.0, color="black", linewidth=1)
axes[2].set_title("Net Beta Notional")
axes[2].set_ylabel("Beta Notional")
axes[2].set_xlabel("Date")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Optional quick look at first rebalances
bt.loc[bt["rebalance"], ["a_pos_notional", "b_pos_notional", "beta_notional", "equity"]].head(10)

KeyError: ('weekly', 'MSTZ', 'MSTR')